In [3]:
 # 計算によく使うライブラリ
import pandas as pd
import numpy as np
import re 
import os
import glob
from itertools import combinations, product
import math
from math import floor

# 視覚化のためのライブラリ（必ずしも必要ではない）
import matplotlib.pyplot as plt

# 結果データの保存のためのライブラリ
import pickle

# samtoolsの機能を使うためのライブラリ
import pysam

#　計算時間の表示のためのライブラリ
from datetime import datetime, timedelta

#　マルチスレッドで並行処理をする場合に使うライブラリ
from threading import Thread, Lock

#　不要な警告を表示しないためのライブラリ（必ずしも必要ではない）
import warnings
warnings.filterwarnings("ignore")


def cd(usb):
    if usb == 0:
        %cd 
    elif usb == 1:
        %cd /Volumes/GENDATAENV1/
    elif usb == 2:
        %cd /Volumes/GENDATAENV2/
    elif usb == 3:
        %cd /Volumes/GENDATAENV3/
        
class timereporter:
    def __init__(self, total):
        self.total = total
    
    def start(self):
        self.starttime = datetime.now()
        print(f"Session starting at {datetime.now()}")
        
    def report(self, count):
        time_until_now = (datetime.now()-self.starttime).total_seconds()
        time_per_count = count
        linear_forecast = (time_until_now/count) * (self.total-count)
        print(f"{100*count/self.total}% done. Expecting {linear_forecast} more seconds")
    def end(self):
        print(f"Session completed after {(datetime.now()-self.starttime).total_seconds()} seconds at {datetime.now()}")
        

In [1]:
class GWAS:
    def __init__(self, raw_file_path: str):
        self.raw_file_path = raw_file_path
        self.data = pd.DataFrame()
    
    def info(self, header="infer", nrows=10, compression="infer", sep="\t"):
        head = pd.read_table(self.raw_file_path, nrows=nrows, header=header, compression=compression, sep=sep)
        print(head.columns)
        print(head.isna().sum())
        return head
        
    def load(self, header="infer", use_cols=None, dropNA=True, dropDUP=True, dropDUPcol=["hm_chrom", "hm_pos", "hm_effect_allele"]):
        self.data = pd.read_table(self.raw_file_path, header=header, usecols=use_cols)
        if dropNA:
            self.data.dropna(axis=0, inplace=True)
        if dropDUP:
            self.data.drop_duplicates(subset=dropDUPcol, inplace=True)
        self.data.reset_index(inplace=True, drop=True)

    def setcols(self, include_cols= ["hm_chrom", "hm_pos", "hm_beta", "hm_effect_allele_frequency", "hm_effect_allele", "p_value", "hm_other_allele", "standard_error"], mapper = {"hm_chrom":"chrom", 
                                                                                                                                                                                  "hm_pos":"pos", 
                                                                                                                                                                                  "hm_beta":"beta", 
                                                                                                                                                                                  "hm_effect_allele_frequency": "eaf",
                                                                                                                                                                                  "hm_effect_allele": "eff",
                                                                                                                                                                                  "p_value": "pval",
                                                                                                                                                                                  "hm_other_allele": "non-eff",
                                                                                                                                                                                  "standard_error": "std"}):
        self.data = self.data[include_cols]
        self.data.rename(columns=mapper, inplace=True) 
    
    def chrom_to_string(self, chrom_col="chrom"):
        if "chr" not in str(self.data[chrom_col][0]):
            string_chrom = self.data[chrom_col].apply(lambda x: "chr"+str(x).split(".")[0].split("_")[0].upper())
            self.data[chrom_col] = string_chrom

    def add_ref(self, fasta_path="/Volumes/GENDATAENV/GRCh38/GCF_000001405.40_GRCh38.p14_genomic.fna", 
                GRCh38_dict = {"chr1": "NC_000001.11",
                              "chr2": "NC_000002.12",
                              "chr3": "NC_000003.12",
                              "chr4": "NC_000004.12",
                              "chr5": "NC_000005.10",
                              "chr6": "NC_000006.12",
                              "chr7": "NC_000007.14",
                              "chr8": "NC_000008.11",
                              "chr9": "NC_000009.12",
                              "chr10": "NC_000010.11",
                              "chr11": "NC_000011.10",
                              "chr12": "NC_000012.12",
                              "chr13": "NC_000013.11",
                              "chr14": "NC_000014.9",
                              "chr15": "NC_000015.10",
                              "chr16": "NC_000016.10",
                              "chr17": "NC_000017.11",
                              "chr18": "NC_000018.10",
                              "chr19": "NC_000019.10",
                              "chr20": "NC_000020.11",
                              "chr21": "NC_000021.9",
                              "chr22": "NC_000022.11",
                              "chrX": "NC_000023.11",
                              "chrY": "NC_000024.10",
                              "chrMT": "NC_012920.1"}):
        fna = pysam.FastaFile(fasta_path)
        refs = []
        reporter = timereporter(len(self.data))
        reporter.start()
        for index, row in self.data.iterrows():
            chrom = row["chrom"]
            if chrom == "chrM":
                chrom = "chrMT"
            pos = row["pos"]
            ref = fna.fetch(GRCh38_dict[chrom], pos-1, pos)[0]
            refs.append(ref.upper())
            
            if ((index + 1) == 1000) or ((index + 1) == floor(len(self.data) / 2)) or ((index + 1) == floor(len(self.data) / 4)) or ((index + 1) == floor(len(self.data) / 10)) or ((index + 1) == floor(len(self.data) / 100)):
                reporter.report((index + 1))
            
        self.data["ref"] = refs
        reporter.end()
    
    def add_maf(self, eafcol="eaf"):
        mafs = []
        reporter = timereporter(len(self.data))
        reporter.start()
        for index, snp in self.data.iterrows():
            maf = min([snp["eaf"], abs(1-snp["eaf"])])
            mafs.append(maf)
            
            if ((index + 1) == 1000) or ((index + 1) == floor(len(self.data) / 2)) or ((index + 1) == floor(len(self.data) / 4)) or ((index + 1) == floor(len(self.data) / 10)) or ((index + 1) == floor(len(self.data) / 100)):
                reporter.report((index + 1))
            
        self.data["maf"] = mafs
        reporter.end()
        
    def flip_effect(self):
        new_beta = []
        new_eff = []
        reporter = timereporter(len(self.data))
        reporter.start()
        for index, row in self.data.iterrows():
            if row["eff"].upper() == row["ref"]:
                #print(self.data.iloc[index]["eff"], self.data.iloc[index]["non-eff"], self.data.iloc[index]["ref"])
                new_eff.append(self.data.iloc[index]["non-eff"].upper())
                new_beta.append(-1*self.data.iloc[index]["beta"])
            else:
                new_eff.append(self.data.iloc[index]["eff"].upper())
                new_beta.append(self.data.iloc[index]["beta"])
                
            if ((index + 1) == 1000) or ((index + 1) == floor(len(self.data) / 2)) or ((index + 1) == floor(len(self.data) / 4)) or ((index + 1) == floor(len(self.data) / 10)) or ((index + 1) == floor(len(self.data) / 100)):
                reporter.report((index + 1))
                
        self.data["beta"] = new_beta
        self.data["eff"] = new_eff
        reporter.end()
        
    def separate_variants(self, splitter="/"):
        # if the GWAS SNPs are defined using 1 v n-multiple alternative alleles, then split th SNP to n-SNPs with 1v1 correspondence.
        # it should be used before flip_effect and after setcols, chrom_to_string, add_ref, add_maf
    
        # first find the rows that have multiple non-eff choices
        # create list of information for each non-eff choices that represent new rows
        # drop the found rows
        # insert new rows
        index_to_drop = []
        rows_to_insert = []
        
        reporter = timereporter(len(self.data))
        reporter.start()
        
        non_eff_colnum = list(self.data.columns).index("non-eff")
        
        for index, row in self.data.iterrows():
            have_multiple = splitter in row["non-eff"]
            
            if have_multiple:
                index_to_drop.append(index)
                non_eff_choices = row["non-eff"].upper().split(splitter)
                
                for non_eff in non_eff_choices:
                    new_row = row.to_list()
                    new_row[non_eff_colnum] = non_eff
                    rows_to_insert.append(new_row)
                
            if ((index + 1) == 1000) or ((index + 1) == floor(len(self.data) / 2)) or ((index + 1) == floor(len(self.data) / 4)) or ((index + 1) == floor(len(self.data) / 10)) or ((index + 1) == floor(len(self.data) / 100)):
                reporter.report((index + 1))     
                
        print(f"iteration finished at {datetime.now()}. moving on to dropping {len(index_to_drop)} rows")
        
        self.data.drop(index=index_to_drop, axis=0, inplace=True)
        self.data.reset_index(inplace=True, drop=True)
        
        print(f"dropping finished at {datetime.now()}. moving on to insertion of {len(rows_to_insert)} new rows")
        
        insert_df = pd.DataFrame(rows_to_insert, columns=self.data.columns)
        self.data = pd.concat([self.data, insert_df])
            
        print(f"insertion phase finished at {datetime.now()}. Next, sort and clean the dataframe.")
        
        order = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY", "chrXY", "chrMT", "chrM"]
        self.data["chrom"] = pd.Categorical(self.data["chrom"], order)
        self.data.sort_values(["chrom"], inplace=True)
        self.data.reset_index(inplace=True, drop=True)
        reporter.end()
    
    

In [ ]:
class LDdata:
    def __init__(self, raw_file_path_dict=None, default=False):
        if default:
            raw_file_path_dict = {}
            for chrom in [f"chr{i}" for i in range(1, 23)]:
                raw_file_path_dict[chrom] = f"/Volumes/GENDATAENV/tommo54k/tommo_cooccurrence/tommo-54kjpn-20230828-GRCh38-autosome-{chrom}-plink-r2.tsv.gz"
            raw_file_path_dict["chrXY"] = f"/Volumes/GENDATAENV/tommo54k/tommo_cooccurrence/tommo-54kjpn-20230828-GRCh38-chrXY_PAR2-chrX-plink-r2.tsv.gz"
            raw_file_path_dict["chrMT"] = f"/Volumes/GENDATAENV/tommo54k/tommo_cooccurrence/tommo-54kjpn-20230828-GRCh38-mitochondria-chrM-plink-r2.tsv"
        self.dict = raw_file_path_dict
    
    def set_fetcher(self, chrom: str):
        self.pysam_fetcher = pysam.TabixFile(self.dict[chrom])
    
    def fetcher(self):
        return self.pysam_fetcher
    

In [ ]:
class PGS:
    def __init__(self, vcf_raw_paths: list, gwasdata, ld=None, TOMMO_gt_paths=["/Volume/GENDATAENV1/tommo54k/gt/tommo-54kjpn-20230626r2-GRCh38-gf-autosome.vcf.gz", "/Volume/GENDATAENV1/tommo54k/gt/tommo-54kjpn-20230626r2-GRCh38-gf-chrX_PAR2.vcf.gz"], target_samples=["all"]): #[f"NA{i}" for i in range(18939,19092)]
        self.variant_fetchers = dict()
        for file in vcf_raw_paths:
            if type(file)!=list:
                self.variant_fetchers[file] = pysam.VariantFile(file)
            elif type(file)==list:
                file_indicator = ""
                for i in file:
                    file_indicator += i
                self.variant_fetchers[file_indicator] = dict()
                for i in file:
                    self.variant_fetchers[file_indicator][i] = pysam.VariantFile(i)
                
        self.gwas = gwasdata
        
        self.ld = ld
        
        self.vcf_raw_paths_raw = vcf_raw_paths
        
        self.vcf_raw_paths = []
        self.path_type = dict()
        for file in vcf_raw_paths:
            if type(file)!=list:
                self.vcf_raw_paths.append(file)
                self.path_type[file] = str
            if type(file)==list:
                file_indicator = ""
                for i in file:
                    file_indicator += i
                self.vcf_raw_paths.append(file_indicator)
                self.path_type[file_indicator] = list
        
        self.target_samples = target_samples
        
        self.ldfetcher = None
        
        self.tommo_fetchers = dict()
        for file in TOMMO_gt_paths:
            self.tommo_fetchers[file] = pysam.VariantFile(file)
        
        self.scores_df = None
        
        self.scores = dict() 
        for file in vcf_raw_paths:
            j = vcf_raw_paths.index(file)
            if type(file)!=list:
                if target_samples[j]=="all":
                    self.scores[file] = {sample: {"pgs":0, "variant count": 0} for sample in self.variant_fetchers[file].header.samples}
                else:
                    self.scores[file] = {sample: {"pgs":0, "variant count": 0} for sample in target_samples[j]}
            elif type(file)==list:
                file_indicator = ""
                for i in file:
                    file_indicator += i
                if target_samples[j]=="all":
                    self.scores[file_indicator] = {sample: {"pgs":0, "variant count": 0} for sample in self.variant_fetchers[file_indicator][file[0]].header.samples}
                else:
                    self.scores[file_indicator] = {sample: {"pgs":0, "variant count": 0} for sample in target_samples[j]}
            
        self.metrics = {
            "pval": None,
            "maf": None,
            "info": None,
            "hwe": None,
            "r2": None,
            "kilobase": None,
        }
        
    def  clump(self, r2=0.5, kilobase=1000):
        if (len(self.gwas)>0):
            print("Start LD clumping.")
            self.metrics["r2"] = r2
            self.metrics["kilobase"] = kilobase

            index_to_drop = []

            reporter = timereporter(len(self.gwas))
            reporter.start()

            prev_chrom = 0

            for index, snp in self.gwas.iterrows():
                chrom = snp["chrom"]
                if (chrom=="chrX")|(chrom=="chrY")|(chrom=="chrx")|(chrom=="chry"):
                    chrom = "chrXY"
                if (chrom=="chrM")|(chrom=="chrm")|(chrom=="chrmt"):
                    chrom = "chrMT"

                if prev_chrom != chrom:
                    self.ld.set_fetcher(chrom)
                    self.ldfetcher = self.ld.fetcher()

                prev_chrom = chrom

                if chrom=="chrMT":
                    chrom = "chrM"
                if chrom=="chrXY":
                    chrom = "chrX"

                pos = snp["pos"]
                window_lower = pos-kilobase*1000
                window_upper = pos+kilobase*1000

                snp_r2 = 0
                num_clumped = 0
                # while looping through ld data, if fetched snp is in the snp list, then add r2 to the loop invariant snp 
                for i in self.ldfetcher.fetch(chrom, pos-1, pos):
                    retrieved = re.split(r'\t+', i.rstrip('\n'))
                    pair_r2 = float(retrieved[12])
                    pair_pos = float(retrieved[8])
                    # if not self and is within window
                    if (pos != pair_pos)&(window_lower<pos<window_upper):
                        if pair_pos in self.gwas["pos"]:
                            snp_r2 += pair_r2

                    # if snp_r2 exceeds threshold, delete the snp from list and break
                    if snp_r2 > r2:
                        index_to_drop.append(index)
                        break

                if ((index + 1) == 1000) or ((index + 1) == floor(len(self.gwas) / 2)) or ((index + 1) == floor(len(self.gwas) / 4)) or ((index + 1) == floor(len(self.gwas) / 10)) or ((index + 1) == floor(len(self.gwas) / 100)):
                    reporter.report((index + 1))

            self.gwas.drop(index=index_to_drop, axis=0, inplace=True)
            num_clumped = len(index_to_drop)
            print(f"{num_clumped} variants excluded by LD.")
            reporter.end()
            self.gwas.reset_index(inplace=True, drop=True)

    def plot_by_pval(self, num_=300, pval_col="pval", space=8):
        gwas_df = self.gwas
        # Define a range of p-value levels to explore
        pval_levels = np.logspace(-space, 0, num=num_)  # Example: from 1e-8 to 1, logarithmically spaced

        # Initialize a dictionary to hold the counts of variants for each p-value level
        variant_counts = {}

        # Loop through each p-value level, filter the DataFrame, and count the variants
        for pval in pval_levels:
            filtered_df = gwas_df[gwas_df[pval_col] <= pval]
            count = filtered_df.shape[0]
            variant_counts[pval] = count

        # Plotting
        plt.figure(figsize=(10, 6))
        # When plotting from a dictionary, use `.keys()` for the x-axis and `.values()` for the y-axis
        plt.plot(list(variant_counts.keys()), list(variant_counts.values()), marker='o', linestyle='-')
        plt.xscale('log')  # Use a logarithmic scale for the x-axis to better display the range of p-values
        plt.xlabel('P-value Level')
        plt.ylabel('Number of Variants')
        plt.title('Number of Variants by P-value Level in GWAS Results')
        plt.grid(True, which="both", ls="--")
        plt.show()
        print(variant_counts)

            
    def threshold(self, pval_thr=None, maf_thr=None, info_thr=None, hwe_thr=0.05):
        print("Start pvalue, MAF, imputation, HWE thresholding.")
        self.metrics["pval"] = pval_thr
        self.metrics["maf"] = maf_thr
        self.metrics["info"] = info_thr
        self.metrics["hwe"] = hwe_thr
        
        if bool(pval_thr):
            original_length = len(self.gwas)
            self.gwas = self.gwas[self.gwas["pval"]<pval_thr]
            print(f"{original_length - len(self.gwas)} variants excluded by pvalue.")
        if bool(maf_thr):
            original_length = len(self.gwas)
            self.gwas = self.gwas[self.gwas["maf"]>maf_thr]
            print(f"{original_length - len(self.gwas)} variants excluded by MAF.")
        if bool(info_thr):
            original_length = len(self.gwas)
            self.gwas = self.gwas[self.gwas["info"]>info_thr]
            print(f"{original_length - len(self.gwas)} variants excluded by imputation quality.")
        self.gwas.reset_index(inplace=True, drop=True)
        if bool(hwe_thr) & (len(self.gwas)>0):
            # fetch snp rom vcf, look up hwe pvalue, and apply threshold
            index_to_drop = []
            
            reporter = timereporter(len(self.gwas))
            reporter.start()
            
            for index, snp in self.gwas.iterrows():
                chrom = snp["chrom"]
                pos = snp["pos"]
                eff = snp["eff"].upper()
                for fetcher in self.tommo_fetchers.values():
                    for record in fetcher.fetch(chrom, pos-1, pos):
                        retrieved = re.split(r'\t', str(record).rstrip('\n'))
                        #print(retrieved)
                        hwe = list(map(float, retrieved[-1].split("HWE")[-1].split("=")[1].split(";")[0].split(",")))
                        alt = retrieved[4].split(",")
                        try:
                            which_alt = alt.index(eff)
                        except ValueError:
                            break
                        #print(alt)
                        #print(which_alt)
                        try:
                            hwe_pval = hwe[which_alt]
                        except IndexError:
                            print("error:", hwe, which_alt, alt)
                        hwe_pval = hwe[which_alt]
                        #print(hwe, which_alt, hwe_pval, type(hwe_pval))
                        
                        if hwe_pval > hwe_thr:
                            index_to_drop.append(index)
                            
                if ((index + 1) == 1000) or ((index + 1) == floor(len(self.gwas) / 2)) or ((index + 1) == floor(len(self.gwas) / 4)) or ((index + 1) == floor(len(self.gwas) / 10)) or ((index + 1) == floor(len(self.gwas) / 100)):
                    reporter.report((index + 1))
                        
            self.gwas.drop(index=index_to_drop, axis=0, inplace=True)
            num_excluded = len(index_to_drop)
            print(f"{num_excluded} variants excluded by HWE.")
        
            reporter.end()
            self.gwas.reset_index(inplace=True, drop=True)

    def snplist(self, head=False):
        return self.gwas
        
    def compute(self):
        """
        Procedure:
        1. create score memory for each input vcf files and each samples
        2. go through gwas snp list and fetch vcf
        3. verify that the matched variants have same reference allele
        4. find which alt allele is the targetted snp
        5. extract genotype information
        6. find the appropriate PGS allele count weight
        7. compute PGS based on beta and PGS allele count weight
        8. add computed PGS to the memory of the sample
        9. increment variant count
        """
        
        reporter = timereporter(len(self.gwas))
        reporter.start()
        
        for i, snp in self.gwas.iterrows():
            chrom = snp["chrom"]
            pos = snp["pos"]
            ref = snp["ref"]
            eff = snp["eff"].upper()
            beta = snp["beta"]
            for index, path in enumerate(self.vcf_raw_paths):
                if self.path_type[path] != list:
                    for record in self.variant_fetchers[path].fetch(chrom, pos-1, pos):
                        if record.ref == ref: # check just in case
                            try: 
                                which_alt = 0 if eff==record.ref else record.alts.index(eff) + 1
                            except ValueError:
                                break
                            for sample in self.scores[path].keys():
                                genotype = record.samples[sample]["GT"] # expecting a tuple from pysam
                                if None in genotype:
                                    break
                                else:
                                    eff_count = genotype.count(which_alt)
                                    snp_pgs = beta * eff_count
                                    self.scores[path][sample]["pgs"] += snp_pgs
                                    if eff_count > 0:
                                        self.scores[path][sample]["variant count"] += 1


                elif self.path_type[path] == list: 
                    for sub_path in self.vcf_raw_paths_raw[index]:
                        try:
                            for record in self.variant_fetchers[path][sub_path].fetch(chrom, pos-1, pos):
                                if record.ref == ref: # check just in case
                                    try: 
                                        which_alt = 0 if eff==record.ref else record.alts.index(eff) + 1
                                    except ValueError:
                                        break
                                    for sample in self.scores[path].keys():
                                        genotype = record.samples[sample]["GT"] # expecting a tuple from pysam
                                        if None in genotype:
                                            break
                                        else:
                                            eff_count = genotype.count(which_alt)
                                            snp_pgs = beta * eff_count
                                            self.scores[path][sample]["pgs"] += snp_pgs
                                            if eff_count > 0:
                                                self.scores[path][sample]["variant count"] += 1
                        except ValueError:
                            continue
            
            if ((i+1) == 1000) or ((i+1) == floor(len(self.gwas) / 2)) or ((i+1) == floor(len(self.gwas) / 4)) or ((i+1) == floor(len(self.gwas) / 10)) or ((i+1) == floor(len(self.gwas) / 100)):
                reporter.report((i+1))
        
        reporter.end()

                                
    def scores_to_dataframe(self):
        sampleids = []
        pgss = []
        vcs = []
        vcfs = []
        for key in self.scores:
            if "pgs" not in self.scores[key]:
                for subkey in self.scores[key]:
                    pgs = self.scores[key][subkey]["pgs"]
                    vc = self.scores[key][subkey]['variant count']
                    sampleid = subkey
                    vcf = key
                    pgss.append(pgs)
                    vcs.append(vc)
                    sampleids.append(sampleid)
                    vcfs.append(vcf)
                
        self.scores_df = pd.DataFrame({
            "SampleID": sampleids,
            "PolygenicRiskScore": pgss,
            "VariantCount": vcs,
            "SourceFile": vcfs
        })
                
            